# 05 — Supervision : valider les tonalités du dunun (V2)

But : passer d'une hypothèse (mon clustering ouvert/fermé) à un **résultat scientifique**, en s'appuyant sur **ta vérité terrain** (annotation à l'oreille, à l'aveugle).

**Avant de lancer** : ouvre l'outil d'annotation, étiquette les 150 frappes, exporte, et enregistre le fichier sous **`output/dunun_labels.csv`** (colonnes `id,label`).

Ce notebook mesure alors :
1. l'**accord** entre ton oreille et le clustering (matrice, accuracy, indice ARI) ;
2. la part d'**incertain** (si tu hésites beaucoup, la catégorie est floue) ;
3. un **classifieur supervisé** entraîné sur TES labels (validation croisée) → la métrique honnête ;
4. application à toutes les frappes avec un **niveau de confiance**.

## 0. Charger ton annotation

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import utils_V2 as u

SAMPLE = 'output/dunun_annot_sample.csv'   # produit par l'outil (avec la prédiction cachée)
LABELS = 'output/dunun_labels.csv'         # TON export d'annotation
assert os.path.exists(SAMPLE), 'Lance d\'abord annotation_dunun_V2.build().'
if not os.path.exists(LABELS):
    raise SystemExit('⚠️  Annote d\'abord (artifact) puis enregistre tes labels sous output/dunun_labels.csv')

sample = pd.read_csv(SAMPLE)                # id, features, SD, piece, take, code, tonalite (prédite)
labels = pd.read_csv(LABELS)               # id, label (humain)
m = sample.merge(labels, on='id', how='inner')
print(f'{len(m)} frappes annotées sur {len(sample)}')
print(m['label'].value_counts().to_string())

## 1. Es-tu constant ? — la part d'« incertain »
Si tu as souvent hésité, c'est que la frontière ouvert/fermé est floue (info en soi).

In [ ]:
part_inc = (m['label']=='incertain').mean()
print(f"Incertain : {part_inc*100:.0f}% des frappes")
print('  < 15% : catégorie nette | 15-35% : correcte mais floue aux bords | > 35% : catégorie fragile')

## 2. Ton oreille valide-t-elle mon clustering ?
On compare TON label au groupe prédit (sur les frappes non-incertaines).

In [ ]:
from sklearn.metrics import confusion_matrix, adjusted_rand_score, accuracy_score
cert = m[m['label']!='incertain'].copy()
labs = ['ouvert','fermé']
cm = confusion_matrix(cert['label'], cert['tonalite'], labels=labs)
acc = accuracy_score(cert['label'], cert['tonalite'])
ari = adjusted_rand_score(cert['label'], cert['tonalite'])
print(f'Accord oreille ↔ clustering : {acc*100:.0f}%   |   ARI = {ari:.2f}')
print('  (ARI : 0 = hasard, 1 = accord parfait)')
display(pd.DataFrame(cm, index=['oreille '+l for l in labs], columns=['cluster '+l for l in labs]))

## 3. Classifieur SUPERVISÉ sur tes labels (validation croisée)
La vraie métrique scientifique : un modèle entraîné sur ta vérité terrain, testé sur des frappes qu'il n'a pas vues.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
X = cert[u.FAST_FEATS].values; y = cert['label'].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)
pred = cross_val_predict(RandomForestClassifier(300, random_state=0), X, y, cv=cv)
print(f'Classifieur supervisé (5-fold) : {accuracy_score(y, pred)*100:.0f}% sur {len(y)} frappes annotées')
display(pd.DataFrame(confusion_matrix(y, pred, labels=labs),
        index=['vrai '+l for l in labs], columns=['préd '+l for l in labs]))
clf = RandomForestClassifier(300, random_state=0).fit(X, y)
imp = pd.Series(clf.feature_importances_, index=u.FAST_FEATS).sort_values()
imp.plot.barh(figsize=(7,3.5), color='#4a8cff'); plt.title('Ce qui distingue ouvert/fermé (selon ton oreille)'); plt.tight_layout(); plt.show()

## 4. Appliquer à TOUTES les frappes, avec confiance

In [ ]:
full = pd.read_csv('output/dunun_frappes_V2.csv')
proba = clf.predict_proba(full[u.FAST_FEATS].values)
full['tonalite_sup'] = clf.classes_[proba.argmax(1)]
full['confiance'] = proba.max(1)
full.to_csv('output/dunun_frappes_V2_supervise.csv', index=False)
print('Appliqué ->', len(full), 'frappes | confiance moyenne :', round(full['confiance'].mean(),2))
print('frappes à haute confiance (>0.8) :', f"{(full['confiance']>0.8).mean()*100:.0f}%")
display(full.groupby('tonalite_sup').size().rename('n'))

## Conclusion (à remplir avec tes chiffres)

> Les frappes de dunun se répartissent en 2 tonalités (**ouvert / fermé**), catégorie validée par annotation humaine à l'aveugle : **incertain __%**, accord oreille↔clustering **__%** (ARI __), classifieur supervisé **__%** en validation croisée.

Selon les résultats :
- **accord élevé + peu d'incertain** → mon clustering était bon, le nom *ouvert/fermé* est justifié. ✅
- **accord moyen** → il y a bien 2 timbres mais la frontière est graduelle ; on garde avec la nuance.
- **beaucoup d'incertain / accord faible** → *ouvert/fermé* n'est pas la bonne grille ; on renomme ou on cherche d'autres classes.

Robustesse en plus : refais annoter par un ami joueur → **accord inter-annotateur** (kappa de Cohen).